# 05.2 XGBoost Root-Cause Sensitivity

Notebook 05 explains the selected `random_forest` model. This companion notebook asks a narrower robustness question: because XGBoost was close to random forest in cross-validated PR-AUC, do the leading sensor rankings hold if explanations are generated from the near-tie XGBoost model instead?

The selected production artifact remains `models/selected_model.joblib`; this notebook is a robustness check on the sensor ranking, not a second model-selection step.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

sys.path.insert(0, str(ROOT / "src"))

from yield_risk.config import load_config
from yield_risk.explainability import (
    compute_shap_values,
    global_feature_importance,
    save_shap_values,
)
from yield_risk.root_cause import (
    fail_shap_lift,
    rank_root_cause_candidates,
    spc_flag_rate,
)

pd.set_option("display.max_colwidth", 120)

In [2]:
cfg = load_config(ROOT / "configs" / "config.yaml")
reports_dir = cfg.paths.reports_dir
models_dir = cfg.paths.models_dir

train = pd.read_csv(cfg.paths.splits_dir / "train.csv")
test = pd.read_csv(cfg.paths.splits_dir / "test.csv")
sensor_cols = [c for c in test.columns if c.startswith("sensor_")]

X_train = train[sensor_cols]
X_test = test[sensor_cols]
y_test = test["label"].to_numpy()

rf_candidates = pd.read_csv(reports_dir / "root_cause_candidates.csv")
rf_global = pd.read_csv(reports_dir / "shap_global_importance.csv")
xgb_pipeline = joblib.load(models_dir / "xgboost.joblib")

print(f"Loaded {len(X_train):,} train rows and {len(X_test):,} test rows.")
print(f"Loaded XGBoost model from {models_dir / 'xgboost.joblib'}")
print(f"Selected-model root-cause baseline: {reports_dir / 'root_cause_candidates.csv'}")

Loaded 1,253 train rows and 314 test rows.
Loaded XGBoost model from models\xgboost.joblib
Selected-model root-cause baseline: reports\root_cause_candidates.csv


## Method

The sensitivity path mirrors notebook 05 exactly, except the fitted model is `models/xgboost.joblib` rather than the selected random-forest artifact. The comparison uses the same processed test rows, the same SHAP utility, the same fail/pass SHAP lift calculation, and the same SPC flag-rate calculation.

In [3]:
xgb_explanations = compute_shap_values(xgb_pipeline, X_train, X_test)
xgb_global = global_feature_importance(xgb_explanations)
xgb_lift = fail_shap_lift(xgb_explanations, y_test)
flag_rates = spc_flag_rate(test, sensor_cols)
xgb_candidates = rank_root_cause_candidates(xgb_global, xgb_lift, flag_rates)

save_shap_values(xgb_explanations, reports_dir / "xgboost_shap_values.npz")
xgb_global.to_csv(reports_dir / "xgboost_shap_global_importance.csv", index=False)
xgb_candidates.to_csv(reports_dir / "xgboost_root_cause_candidates.csv", index=False)

print(f"Computed XGBoost SHAP for {len(X_test):,} test rows and {len(sensor_cols):,} sensors.")
print("Saved XGBoost SHAP and root-cause sensitivity artifacts to reports/.")
xgb_candidates.head(10)

Background dataset has 1253 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1253 when initializing the masker.


Computed XGBoost SHAP for 314 test rows and 198 sensors.
Saved XGBoost SHAP and root-cause sensitivity artifacts to reports/.


,sensor,mean_abs_shap,shap_lift,spc_flag_rate,composite_score
0,sensor_059,0.250681,0.463112,0.019108,0.817944
1,sensor_033,0.280181,0.124638,0.015924,0.639563
2,sensor_519,0.197047,0.113691,0.025478,0.519408
3,sensor_205,0.236246,0.101401,0.003185,0.499046
4,sensor_021,0.190246,0.069680,0.015924,0.443467
5,sensor_573,0.148583,0.012330,0.028662,0.379025
6,sensor_488,0.194093,-0.049036,0.000000,0.378136
7,sensor_160,0.128401,0.012890,0.035032,0.366900
8,sensor_129,0.157285,0.073974,0.000000,0.328604
9,sensor_031,0.177057,0.018089,0.000000,0.327687


## Ranking Overlap

A sensor ranking is more robust when a near-tie model family surfaces many of the same leading sensors. The tables below compare the selected random forest from notebook 05 with the close XGBoost challenger from notebook 04.

In [4]:
def top_sensor_set(df: pd.DataFrame, n: int) -> set[str]:
    """Return the top-n sensor names from a root-cause candidate table."""
    return set(df.head(n)["sensor"].tolist())


summary_rows = []
for n in (5, 10, 20):
    rf_top = top_sensor_set(rf_candidates, n)
    xgb_top = top_sensor_set(xgb_candidates, n)
    overlap = sorted(rf_top & xgb_top)
    union = rf_top | xgb_top
    summary_rows.append(
        {
            "top_n": n,
            "overlap_count": len(overlap),
            "jaccard": len(overlap) / len(union),
            "overlap_sensors": ", ".join(overlap),
        }
    )

sensitivity_summary = pd.DataFrame(summary_rows)
sensitivity_summary.to_csv(
    reports_dir / "root_cause_model_sensitivity_summary.csv", index=False
)

rf_ranked = rf_candidates.reset_index().rename(
    columns={
        "index": "rf_rank_zero_based",
        "composite_score": "rf_composite_score",
        "mean_abs_shap": "rf_mean_abs_shap",
        "shap_lift": "rf_shap_lift",
    }
)
rf_ranked["rf_rank"] = rf_ranked["rf_rank_zero_based"] + 1
xgb_ranked = xgb_candidates.reset_index().rename(
    columns={
        "index": "xgb_rank_zero_based",
        "composite_score": "xgb_composite_score",
        "mean_abs_shap": "xgb_mean_abs_shap",
        "shap_lift": "xgb_shap_lift",
    }
)
xgb_ranked["xgb_rank"] = xgb_ranked["xgb_rank_zero_based"] + 1

detail_cols = [
    "sensor",
    "rf_rank",
    "xgb_rank",
    "rf_composite_score",
    "xgb_composite_score",
    "rf_mean_abs_shap",
    "xgb_mean_abs_shap",
    "rf_shap_lift",
    "xgb_shap_lift",
]
sensitivity_detail = (
    rf_ranked.merge(xgb_ranked, on="sensor", how="outer")
    .assign(best_rank=lambda df: df[["rf_rank", "xgb_rank"]].min(axis=1))
    .sort_values(["best_rank", "sensor"])
    [detail_cols]
)
sensitivity_detail.to_csv(
    reports_dir / "root_cause_model_sensitivity_detail.csv", index=False
)

display(sensitivity_summary)
display(sensitivity_detail.head(15))

,top_n,overlap_count,jaccard,overlap_sensors
0,5,4,0.666667,"sensor_033, sensor_059, sensor_205, sensor_519"
1,10,7,0.538462,"sensor_031, sensor_033, sensor_059, sensor_129, sensor_160, sensor_205, sensor_519"
2,20,12,0.428571,"sensor_021, sensor_031, sensor_033, sensor_059, sensor_129, sensor_160, sensor_175, sensor_205, sensor_460, sensor_5..."


,sensor,rf_rank,xgb_rank,rf_composite_score,xgb_composite_score,rf_mean_abs_shap,xgb_mean_abs_shap,rf_shap_lift,xgb_shap_lift
39,sensor_059,1,1,0.870588,0.817944,0.005736,0.250681,0.010661,0.463112
22,sensor_033,2,2,0.478730,0.639563,0.004237,0.280181,0.001798,0.124638
172,sensor_519,3,3,0.431327,0.519408,0.003236,0.197047,0.001961,0.113691
111,sensor_205,4,4,0.383114,0.499046,0.003943,0.236246,0.000984,0.101401
12,sensor_021,12,5,0.240832,0.443467,0.001922,0.190246,0.000514,0.069680
20,sensor_031,5,10,0.356939,0.327687,0.003974,0.177057,-0.000374,0.018089
170,sensor_510,6,17,0.308605,0.226280,0.002534,0.080131,0.001027,0.037756
195,sensor_573,21,6,0.192886,0.379025,0.000848,0.148583,0.000465,0.012330
44,sensor_064,7,66,0.297348,0.121256,0.002483,0.029468,0.000785,0.015198
162,sensor_488,48,7,0.144677,0.378136,0.001594,0.194093,-0.000204,-0.049036


In [5]:
rf_top5 = rf_candidates.head(5)["sensor"].tolist()
xgb_top5 = xgb_candidates.head(5)["sensor"].tolist()
top10_overlap = int(sensitivity_summary.loc[sensitivity_summary["top_n"] == 10, "overlap_count"].iloc[0])

print("Selected RF top 5:", ", ".join(rf_top5))
print("XGBoost sensitivity top 5:", ", ".join(xgb_top5))
print(f"Top-10 overlap: {top10_overlap}/10")

if rf_top5[0] == xgb_top5[0]:
    print(f"Both model families rank {rf_top5[0]} as the leading candidate.")
else:
    print("The leading candidate changes across model families; treat root-cause rank order as unstable.")

Selected RF top 5: sensor_059, sensor_033, sensor_519, sensor_205, sensor_031
XGBoost sensitivity top 5: sensor_059, sensor_033, sensor_519, sensor_205, sensor_021
Top-10 overlap: 7/10
Both model families rank sensor_059 as the leading candidate.


## Interpretation

The XGBoost sensitivity check supports the selected-model sensor ranking.
Both tree families rank `sensor_059` as the leading candidate, and the top-five
lists overlap on four sensors: `sensor_059`, `sensor_033`, `sensor_519`, and
`sensor_205`. The leading sensors hold across both model families rather than
depending on the random forest alone.

`sensor_059` remains first and the top-k overlap is strong. SECOM sensor
names are anonymous, so the next step is mapping the leading sensor set to tool
history, recipe context, chamber state, maintenance logs, and lot genealogy
before changing process settings.